In [ ]:
# Standard Library
import os, sys, json, math, random, time
from pathlib import Path
from typing import List, Optional, Tuple, Union
from collections import defaultdict

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Data Science
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# DeepChem
import deepchem
from deepchem.molnet import *

# RDKit
from rdkit import Chem

# Torch
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn import MSELoss
from torch.utils.data import DataLoader

# Accelerate
from accelerate import Accelerator

# Transformers
from transformers import (
    AutoModel, AutoModelForMaskedLM, AutoModelForSequenceClassification,
    AutoTokenizer, RobertaModel, RobertaTokenizer, RobertaTokenizerFast,
    RobertaConfig, RobertaPreTrainedModel, DebertaV2Model, DebertaV2Config,
    DebertaTokenizerFast, AutoConfig, Trainer, TrainingArguments,
    default_data_collator, get_scheduler,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.trainer_pt_utils import get_parameter_names

# Tokenizers
from tokenizers import ByteLevelBPETokenizer
from tokenizers.processors import BertProcessing

# ESM
import esm

# Datasets
from datasets import Dataset, DatasetDict, load_from_disk

# Metrics
import sklearn
from sklearn.metrics import root_mean_squared_error, mean_squared_error, roc_auc_score

# Bitsandbytes
import bitsandbytes as bnb

# Misc
import ipynbname


In [ ]:
def get_notebook_name():
    try:
        return ipynbname.name()
    except Exception:
        return "unknown_notebook"

notebook_name = get_notebook_name().split(".")[0]
os.makedirs("result_graphs", exist_ok=True)
print("Notebook name:", notebook_name)


In [ ]:
model_pth = "../Pre-Training/PubchemModelDeberta"
model_pth2 = "../Pre-Training/ProteinModelDeb"


In [ ]:
SAVE_DIR        = "Data3/inputdataMMAT"
TRAIN_SAVE_PATH = os.path.join(SAVE_DIR, "train_datasets")
TEST_SAVE_PATH  = os.path.join(SAVE_DIR, "test_datasets")
TASKS_PATH      = os.path.join(SAVE_DIR, "tasks.json")

mmat_exists = (
    os.path.exists(TRAIN_SAVE_PATH) and
    os.path.exists(TEST_SAVE_PATH)  and
    os.path.exists(TASKS_PATH)
)

if mmat_exists:
    print("MMAT datasets found — loading from disk.")
    mmat_train = dict(load_from_disk(TRAIN_SAVE_PATH))
    mmat_test  = dict(load_from_disk(TEST_SAVE_PATH))
    with open(TASKS_PATH) as f:
        mmat_tasks = json.load(f)
else:
    raise FileNotFoundError(f"MMAT datasets not found under {SAVE_DIR}. Run data-prep first.")


In [ ]:
updated_datasets = {"train": {}, "test": {}}
Datasets_Available = {}

for task_name, ds in mmat_train.items():
    updated_datasets["train"][task_name] = ds

for task_name, ds in mmat_test.items():
    updated_datasets["test"][task_name] = ds

for task_name in mmat_tasks:
    Datasets_Available[task_name] = {
        "problem_type": "regression",
        "num_labels": 1,
        "used_encoders": "multiencoder",
    }

num_labels_list   = [v["num_labels"]   for v in Datasets_Available.values()]
problem_type_list = [v["problem_type"] for v in Datasets_Available.values()]

print("Train tasks:", list(updated_datasets["train"].keys()))
print("num_labels_list:", num_labels_list)

tokenizer  = DebertaTokenizerFast.from_pretrained(model_pth)
tokenizer2 = DebertaTokenizerFast.from_pretrained(model_pth2)
tokenizer


## Model

In [ ]:
class MLTClassificationHead(nn.Module):
    """Per-task classification / regression head."""
    def __init__(self, config, num_labels):
        super().__init__()
        self.dense    = nn.Linear(config.hidden_size, config.hidden_size // 2)
        self.dropout  = nn.Dropout(config.hidden_dropout_prob)
        self.out_proj = nn.Linear(config.hidden_size // 2, num_labels)

    def forward(self, features, **kwargs):
        x = self.dropout(features)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        return self.out_proj(x)


class RobertaMultiTaskModel(RobertaPreTrainedModel):
    def __init__(self, model_path1, model_path2, num_labels_list, problem_type_list):
        config = AutoConfig.from_pretrained(model_path1)
        super().__init__(config)
        self.config  = config
        self.config2 = AutoConfig.from_pretrained(model_path2)

        self.encoder1 = DebertaV2Model.from_pretrained(model_path1)
        self.encoder2 = DebertaV2Model.from_pretrained(model_path2)

        for param in self.encoder1.parameters():
            param.requires_grad = False
        for param in self.encoder2.parameters():
            param.requires_grad = False

        UNFREEZE_LAST_N = 2
        for layer in self.encoder1.encoder.layer[-UNFREEZE_LAST_N:]:
            for param in layer.parameters():
                param.requires_grad = True
        for layer in self.encoder2.encoder.layer[-UNFREEZE_LAST_N:]:
            for param in layer.parameters():
                param.requires_grad = True

        self.dense   = nn.Linear(config.hidden_size * 3, config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

        self.hidden_size  = config.hidden_size
        self.num_tasks    = len(num_labels_list)
        self.classification_heads = nn.ModuleList(
            [MLTClassificationHead(config, num_labels=n) for n in num_labels_list]
        )
        self.num_labels_list   = num_labels_list
        self.problem_type_list = problem_type_list
        self.post_init()

    def forward(
        self,
        input_ids=None, attention_mask=None,
        input_ids2=None, attention_mask2=None,
        input_ids3=None, attention_mask3=None,
        labels_list=None,
        output_attentions=None, output_hidden_states=None,
        return_dict=None, train=False, task_index=0,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        p1 = self.encoder1(
            input_ids=input_ids, attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        ).last_hidden_state[:, 0, :]

        if input_ids2 is not None and attention_mask2 is not None:
            p2 = self.encoder2(input_ids=input_ids2, attention_mask=attention_mask2,
                               return_dict=return_dict).last_hidden_state[:, 0, :]
            p3 = self.encoder2(input_ids=input_ids3, attention_mask=attention_mask3,
                               return_dict=return_dict).last_hidden_state[:, 0, :]
            combined = torch.cat((p1, p2, p3), dim=1)
        else:
            combined = torch.cat((p1, torch.zeros_like(p1), torch.zeros_like(p1)), dim=1)

        combined = self.dropout(combined)
        combined = self.dense(combined)
        combined = F.gelu(combined)
        logits   = self.classification_heads[task_index](combined)

        return {"logits": logits}


In [ ]:
model = RobertaMultiTaskModel(model_pth, model_pth2, num_labels_list, problem_type_list)

frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Frozen    params: {frozen:,}")
print(f"Trainable params: {trainable:,}")
print(model)


## Optimizer & Training Args

In [ ]:
def get_optimizer(model, run_name):
    training_args = TrainingArguments(
        output_dir=f"./models/multitask_model_{run_name}",
        overwrite_output_dir=True,
        learning_rate=5e-5,
        weight_decay=5e-3,
        do_train=True,
        num_train_epochs=40,
        fp16=True,
        per_device_train_batch_size=32,
        gradient_accumulation_steps=2,
    )
    decay_params = get_parameter_names(model, [nn.LayerNorm])
    decay_params = [
        name for name in decay_params
        if "bias" not in name and model.get_parameter(name).requires_grad
    ]
    grouped = [
        {"params": [p for n, p in model.named_parameters() if n in decay_params],
         "weight_decay": training_args.weight_decay},
        {"params": [p for n, p in model.named_parameters() if n not in decay_params],
         "weight_decay": 0.0},
    ]
    optimizer = bnb.optim.AdamW8bit(
        grouped,
        betas=(training_args.adam_beta1, training_args.adam_beta2),
        eps=training_args.adam_epsilon,
        lr=training_args.learning_rate,
    )
    return optimizer, training_args

optimizer, training_args = get_optimizer(model, notebook_name)
optimizer


In [ ]:
ordered_tasks = {
    key: {"problem_type": value["problem_type"], "num_labels": value["num_labels"]}
    for key, value in Datasets_Available.items()
}

for task_index, task in enumerate(ordered_tasks):
    print(task_index, task, ordered_tasks[task]["problem_type"], ordered_tasks[task]["num_labels"])


## Trainer

In [ ]:
class MultiTaskWeightedTrainer:
    def __init__(
        self, model, datasets, ordered_tasks,
        optimizers=(None, None), args=None, **kwargs
    ):
        precision = "fp16" if args.fp16 else ("bf16" if args.bf16 else "no")
        self.accelerator = Accelerator(mixed_precision=precision)
        self.model     = self.accelerator.prepare(model)
        self.optimizer = self.accelerator.prepare(optimizers[0])

        self.batch_size                  = args.per_device_train_batch_size
        self.gradient_accumulation_steps = args.gradient_accumulation_steps
        self.data_collator               = kwargs.get("data_collator", default_data_collator)
        self.ordered_tasks               = ordered_tasks
        self.task_names                  = list(ordered_tasks.keys())

        self.train_dataloaders  = self._prepare_dataloaders(datasets["train"], train=True)
        self.train_dataloaders2 = self._prepare_dataloaders(datasets["train"], train=False)
        self.eval_dataloaders   = self._prepare_dataloaders(datasets["test"],  train=False)

        total_micro_steps      = sum(len(dl) for dl in self.train_dataloaders.values())
        steps_per_epoch        = math.ceil(total_micro_steps / self.gradient_accumulation_steps)
        self.total_train_steps = steps_per_epoch * args.num_train_epochs

        print(f"Total epochs        : {args.num_train_epochs}")
        print(f"Total micro-steps   : {total_micro_steps}")
        print(f"Total train steps   : {self.total_train_steps}")

        self.lr_scheduler = get_scheduler(
            name="linear", optimizer=self.optimizer,
            num_warmup_steps=self.total_train_steps // 10,
            num_training_steps=self.total_train_steps,
        )

        task_sizes              = {t: len(self.train_dataloaders[t]) for t in self.task_names}
        self.total_task_count   = sum(task_sizes.values())
        self.task_probabilities = np.array(list(task_sizes.values()), dtype=np.float32) / self.total_task_count

        print(f"Mixed precision     : {self.accelerator.mixed_precision}")
        print(f"Model device        : {self.model.device}")
        print(f"Task sizes          : {task_sizes}")
        print(f"Task probabilities  : {self.task_probabilities}")

    def _prepare_dataloaders(self, datasets, train=True):
        batch_multiplier = 1 if train else 8
        return {
            task: self.accelerator.prepare(DataLoader(
                dataset,
                batch_size=self.batch_size if (dataset.num_rows < 5000 and train)
                           else self.batch_size * batch_multiplier,
                shuffle=train,
                collate_fn=self.data_collator,
            ))
            for task, dataset in datasets.items()
        }

    def focal_bce_loss(self, logits, targets, weights=None, reduction="mean"):
        probs        = torch.sigmoid(logits)
        bce_loss     = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        gamma        = 2
        focal_weight = ((1 - probs) ** gamma) * targets + (probs ** gamma) * (1 - targets)
        loss = focal_weight * bce_loss * (weights if weights is not None else 1.0)
        return loss.mean() if reduction == "mean" else loss.sum()

    def compute_loss(self, task_index, batch):
        labels    = batch.pop("labels")
        weights   = batch.pop("weights", None)
        task_name = batch.pop("task_name")
        logits    = self.model(**batch)["logits"]
        ptype     = self.ordered_tasks[task_name]["problem_type"]

        if ptype == "multi_label_classification":
            n = self.model.num_labels_list[task_index]
            return self.focal_bce_loss(logits.view(-1, n), labels.view(-1, n), weights)
        elif ptype == "regression":
            n = self.model.num_labels_list[task_index]
            return MSELoss()(logits.squeeze(), labels.squeeze() if n == 1 else labels)
        else:
            raise ValueError(f"Unknown problem type for task: {task_name}")

    def train_epoch(self):
        losses      = defaultdict(list)
        self.model.train()
        task_iters  = {t: iter(self.train_dataloaders[t]) for t in self.task_names}
        global_step = 0
        max_prob    = max(self.task_probabilities)

        while any(v is not None for v in task_iters.values()):
            task_index = np.random.choice(len(self.task_names), p=self.task_probabilities)
            task       = self.task_names[task_index]
            if task_iters[task] is None:
                continue
            try:
                batch = next(task_iters[task])
                batch["task_name"]  = task
                batch["task_index"] = task_index

                scale = math.sqrt(max_prob / self.task_probabilities[task_index])
                loss  = self.compute_loss(task_index, batch)
                losses[task].append(loss.item())

                if global_step % 25 == 0:
                    print(f"Step {global_step}/{self.total_task_count}: {task}  loss={loss.item():.4f}")

                self.accelerator.backward(loss / self.gradient_accumulation_steps * scale)

                if (global_step + 1) % self.gradient_accumulation_steps == 0:
                    self.optimizer.step()
                    self.lr_scheduler.step()
                    self.optimizer.zero_grad()

                global_step += 1
            except StopIteration:
                task_iters[task] = None

        if global_step % self.gradient_accumulation_steps != 0:
            self.optimizer.step()
            self.lr_scheduler.step()
            self.optimizer.zero_grad()

        return {t: float(np.mean(v)) if v else 0.0 for t, v in losses.items()}

    def evaluate_score(self, dataloader, dataset_type, metrics=None):
        if metrics is None:
            metrics = defaultdict(list)
        self.model.eval()

        for task_index, task_name in enumerate(self.task_names):
            predictions, true_labels, nan_indicators = [], [], []
            task_type = self.ordered_tasks[task_name]["problem_type"]

            for batch in dataloader[task_name]:
                labels  = batch.pop("labels")
                nan_ind = batch.pop("weights", None)
                batch   = {k: v.to(self.accelerator.device) for k, v in batch.items()}
                batch["task_index"] = task_index

                with torch.inference_mode():
                    logits = self.model(**batch)["logits"].detach()
                    preds  = logits if task_type == "regression" else torch.sigmoid(logits)

                predictions.append(preds.cpu().numpy())
                true_labels.append(labels.detach().cpu().numpy())
                if nan_ind is not None:
                    nan_indicators.append(nan_ind.cpu().numpy())

            predictions = np.concatenate(predictions, axis=0)
            true_labels = np.concatenate(true_labels, axis=0)
            if nan_indicators:
                nan_indicators = np.concatenate(nan_indicators, axis=0)

            if task_type == "regression":
                rmse    = root_mean_squared_error(true_labels, predictions)
                score   = {"rmse": rmse}
                average = rmse
            else:
                n_labels = self.ordered_tasks[task_name]["num_labels"]
                if n_labels > 1:
                    valid_aucs = []
                    for j in range(n_labels):
                        mask = nan_indicators[:, j] != 0
                        if len(np.unique(true_labels[mask, j])) > 1:
                            valid_aucs.append(float(roc_auc_score(true_labels[mask, j], predictions[mask, j])))
                        else:
                            valid_aucs.append(None)
                    score   = {"roc_auc": np.array(valid_aucs)}
                    average = np.nanmean([v for v in valid_aucs if v is not None])
                else:
                    auc     = roc_auc_score(true_labels, predictions, average=None)
                    score   = {"roc_auc": auc}
                    average = auc

            metrics[task_name].append(score)
            print(f"Eval [{dataset_type}] {task_name}: {average:.4f}  {score}")

        return metrics

    def train(self, num_epochs, eval_metrics=None, train_metrics=None, notebook_name=""):
        for epoch in range(num_epochs):
            epoch_start = time.time()
            torch.cuda.empty_cache()
            print(f"\n── Epoch {epoch + 1}/{num_epochs} ──")

            train_losses = self.train_epoch()
            epoch_mins   = (time.time() - epoch_start) / 60
            print(f"Epoch {epoch + 1} done in {epoch_mins:.1f} min | losses: {train_losses}")
            torch.cuda.empty_cache()

            # Save checkpoint every 5 epochs
            if (epoch + 1) % 5 == 0:
                save_dir = f"my_trained_model_{notebook_name}/epoch_{epoch + 1}"
                os.makedirs(save_dir, exist_ok=True)
                self.model.save_pretrained(save_dir)
                torch.save({"epoch": epoch + 1, "optimizer": self.optimizer.state_dict()},
                           os.path.join(save_dir, "optimizer_state.pt"))
                torch.save(self.lr_scheduler.state_dict(),
                           os.path.join(save_dir, "scheduler_state.pt"))
                print(f"Checkpoint saved → {save_dir}")

            eval_metrics = self.evaluate_score(self.eval_dataloaders, "eval", eval_metrics)

        train_metrics = self.evaluate_score(self.train_dataloaders2, "train", train_metrics)
        print(f"Train metrics: {train_metrics}")
        print(f"Eval metrics : {eval_metrics}")
        return train_metrics, eval_metrics


## Instantiate & Train

In [ ]:
torch.cuda.empty_cache()

trainer = MultiTaskWeightedTrainer(
    model=model,
    datasets=updated_datasets,
    ordered_tasks=ordered_tasks,
    args=training_args,
    optimizers=(optimizer, None),
)


In [ ]:
# Optional: evaluate before any training to get a baseline
eval_metrics_start = None
eval_metrics_start = trainer.evaluate_score(trainer.eval_dataloaders, "eval")


In [ ]:
eval_metrics  = None
train_metrics = None

train_metrics, eval_metrics = trainer.train(
    num_epochs=int(training_args.num_train_epochs),
    train_metrics=train_metrics,
    eval_metrics=eval_metrics,
    notebook_name=notebook_name,
)


In [ ]:
train_metrics

In [ ]:
eval_metrics

In [ ]:
len(eval_metrics['epigenetic_regulator'])

In [ ]:
for task_name, records in eval_metrics.items():
    print(task_name, len(records))

In [ ]:
eval_metrics_2 = {}

for task_name, start_records in eval_metrics_start.items():
    if task_name not in eval_metrics_2:
        eval_metrics_2[task_name] = []
    eval_metrics_2[task_name] += start_records 

for task_name, eval_records in eval_metrics.items():
    if task_name not in eval_metrics_2:
        eval_metrics_2[task_name] = [] 
    eval_metrics_2[task_name] += eval_records  

In [ ]:
eval_metrics_2

In [ ]:
eval_metrics_2

In [ ]:
eval_metrics_2

## Results & Plots

In [ ]:
def extract_averages_by_metric(nested_dict):
    metrics = {}
    for task_name, records in nested_dict.items():
        for record in records:
            for metric, value in record.items():
                if metric not in metrics:
                    metrics[metric] = {}
                if task_name not in metrics[metric]:
                    metrics[metric][task_name] = []
                if isinstance(value, np.ndarray):
                    metrics[metric][task_name].append(value.mean())
                else:
                    metrics[metric][task_name].append(value)
    return metrics

metrics_data = extract_averages_by_metric(eval_metrics_2)

color_list = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
              "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

for metric, tasks in metrics_data.items():
    plt.figure(figsize=(17.69, 8.27))
    for i, (task_name, values) in enumerate(tasks.items()):
        color = color_list[i % len(color_list)]
        plt.plot(range(len(values)), values, label=task_name, color=color)
    
    if metric == "rmse":
        clipped_values = [v for task_values in tasks.values() for v in list(task_values)[1:]]
        if clipped_values:
            plt.ylim(0, 1.5)
            plt.yticks(np.arange(0, 2.51, 0.5))
    
    plt.title(f"Metric: {metric} — eval over epochs")
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True)
    max_entries = max(len(v) for v in tasks.values())
    plt.xticks(ticks=range(0, max_entries, 10))
    filename = f"{metric}_eval_{notebook_name}.pdf"
    save_path = os.path.join("result_graphs", filename)
    plt.savefig(save_path, format="pdf", dpi=300, bbox_inches="tight")
    print(f"Plot saved → {save_path}")
    plt.show()

In [ ]:
REGRESSION_TASKS = {"lipo", "delaney", "freesolv", "kiba", "davis"}

best_scores = {}
for task_name, records in eval_metrics.items():
    is_regression = task_name in REGRESSION_TASKS or \
                    ordered_tasks.get(task_name, {}).get("problem_type") == "regression"

    per_metric_best = {}

    for epoch_id, record in enumerate(records):
        for metric, value in record.items():
            score = float(np.mean(value)) if isinstance(value, np.ndarray) else float(value)

            if metric not in per_metric_best:
                per_metric_best[metric] = {
                    "best_value": float("inf") if is_regression else float("-inf"),
                    "epoch": None
                }

            current_best = per_metric_best[metric]["best_value"]
            is_better = (is_regression and score < current_best) or \
                        (not is_regression and score > current_best)

            if is_better:
                per_metric_best[metric] = {"best_value": score, "epoch": epoch_id}

    best_scores[task_name] = per_metric_best

    for metric, info in per_metric_best.items():
        print(f"{task_name:30s}  metric={metric:20s}  best={info['best_value']:.4f}  epoch={info['epoch']}")


In [ ]:
for metric, tasks in metrics_data.items():
    plt.figure(figsize=(17.69, 8.27))

    for i, (task_name, values) in enumerate(tasks.items()):
        color = color_list[i % len(color_list)]
        plt.plot(range(len(values)), values, label=f"{task_name} (avg)", color=color)

        if task_name in best_scores and metric in best_scores[task_name]:
            best_value = best_scores[task_name][metric]["best_value"]
            best_epoch = best_scores[task_name][metric]["epoch"]
            plt.scatter(best_epoch, best_value, color=color, s=100, zorder=5,
                        label=f"{task_name} (best)")
            plt.annotate(
                f"Best: {best_value:.3f}",
                (best_epoch, best_value),
                textcoords="offset points",
                xytext=(0, 10),
                ha="center",
                color=color
            )
    plt.yticks(np.arange(0, 2.51, 0.5))
    plt.ylim(0, 2.5)
    plt.title(f"Metric: {metric} — eval over epochs (with best values)")
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True)
    max_entries = max(len(v) for v in tasks.values())
    plt.xticks(ticks=range(0, max_entries, 10))
    filename = f"{metric}_eval_best_{notebook_name}.pdf"
    save_path = os.path.join("result_graphs", filename)
    plt.savefig(save_path, format="pdf", dpi=300, bbox_inches="tight")
    print(f"Plot saved → {save_path}")
    plt.show()

## Save Final Model

In [ ]:
save_dir = f"my_trained_model_{notebook_name}"
os.makedirs(save_dir, exist_ok=True)

trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model saved → {save_dir}")
